# 02 · Teach with examples
An adapter learns from pairs: **an observation → the answer we want**.

| Component | Job in this lesson |
|---|---|
| **OpenClaw + lightweight Nano** | Read the skill, choose tools, coordinate the work. |
| **Python tools** | Build examples, assign rule-based labels, and check the files. |
| **Lightning base** | Write a fresh observation when the tool asks for new wording. |

**Nano orchestrates. Lightning handles the model task. Python keeps labels consistent.**

We will build a small practice dataset with **3 of the workshop’s 31 categories**. These fictional examples teach the mechanics; they are not the full adapter-training dataset.

###  supervised examples and loss masks

the Transformers [chat-template guide](https://huggingface.co/docs/transformers/main/en/chat_templating) explains how role messages become model input. Use the [SDK map](README.md#concept-sdks-and-documentation) for our tested versions.

this lesson constructs 24 labeled training examples and three separate validation examples. Run the data checks, inspect a record, then request one candidate observation from Lightning. The candidate stays outside the dataset until reviewed. Token inspection requires the cached Lightning tokenizer; it does not load model weights.

**Follow the data:** [training_data.py](agent/training_data.py) supplies bulk example generation and displays; the cells below construct individual records and the loss mask. The agent uses [the fixed lesson source](agent/lessons/02-training-data.json). Rule consistency and exact-text separation are checked; semantic coverage is not established.

this lesson illustrates Lightning's real chat tokenization. The CPU model in 03–06 instead uses a small byte vocabulary and an `Observation: … Answer:` format. Both mask context from the loss, but they do not share token IDs or model weights.

**NVIDIA source:** the [Lightning model card](https://build.nvidia.com/nvidia/nemotron-3.5-lightning-30b-a3b/modelcard) identifies the model family and precision. This lesson uses its cached NVFP4 tokenizer; the later CPU model uses a different vocabulary. Tokenizer/template revision must be recorded with a reproducible dataset.


**Manual experiment versus agent reference:** the executable cells below are yours to edit. The OpenClaw section at the end runs the fixed workshop recipe in `agent/lessons/`; it does not ingest your edited cells or reuse manual artifacts. Setup, displays, artifact bookkeeping, and terminal/process transport remain helpers.

### Start here
Load the small display and dataset helpers. The examples and model calls below run in Jupyter.

In [1]:
from pathlib import Path
import sys, json
from IPython.display import JSON, display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course/common.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.common import chat
from course.agent.training_data import (
    build_examples, save_examples, show_pair,
)

### 1 · Show the model the answer we want
This observation comes from one clear camera. With no independent corroboration, its label is `PROBABLE`.

Run the cell to see the observation and expected answer side by side.

In [2]:
ROUTES = {'civil_gathering': 'CIMIC', 'bridge_damage': 'J3-CURRENT-OPS', 'humanitarian_convoy': 'CIMIC'}
SYSTEM = 'Return activity_type, sensor_quality, confidence, and routing as JSON. One source cannot establish CONFIRMED. DEGRADED or UNKNOWN quality requires POSSIBLE.'

In [3]:
def make_label(activity, quality):
    return {'activity_type':activity, 'sensor_quality':quality,
            'confidence':'PROBABLE' if quality=='CLEAR' else 'POSSIBLE',
            'routing':ROUTES[activity]}

def make_record(observation, activity, quality):
    return {'messages':[{'role':'system','content':SYSTEM},
                        {'role':'user','content':observation},
                        {'role':'assistant','content':json.dumps(make_label(activity,quality))}]}

In [4]:
example = make_record(
    "People gather around market stalls. The image is clear. "
    "Only one camera reports this; there is no independent corroboration.",
    "civil_gathering", "CLEAR",
)
show_pair(example)

### 2 · Change the evidence, change the label
Fog makes the image `DEGRADED`. For this exercise, that lowers confidence to `POSSIBLE`.

The activity stays the same. The label should reflect the evidence quality.

In [5]:
fog_example = make_record(
    "People gather around market stalls. The image is degraded by fog. "
    "Only one camera reports this; there is no independent corroboration.",
    "civil_gathering", "DEGRADED",
)
show_pair(fog_example)

### 3 · Build a small dataset
| Observation | Category |
|---|---|
| People at market stalls | `civil_gathering` |
| A visibly broken bridge | `bridge_damage` |
| Trucks delivering relief supplies | `humanitarian_convoy` |

**3 categories × 4 descriptions × 2 image qualities = 24 training examples.**

We also keep **3 separately written validation examples** out of training. All examples here have one source; none should be `CONFIRMED`.

In [6]:
training, validation = build_examples()
report = save_examples(training, validation, ROOT / "course/outputs/preview02/student-data")
display(JSON(report, expanded=True))

<IPython.core.display.JSON object>

### 4 · Inspect the actual training JSON
Each record has three messages: **instructions → observation → expected answer**.

Change `example_number` to inspect another record (0–23). These are real records from the dataset you just built.

In [7]:
example_number = 0
show_pair(training[example_number])
display(JSON(training[example_number], expanded=True))

<IPython.core.display.JSON object>

### 5 · Let Lightning suggest new wording
Here is where we use a model, instead of a fixed data-building rule.

**Known facts → Lightning base → candidate observation → human review**

We supply the facts. Lightning writes the wording. The labels still come from our rules—not from guessing what the model meant. This candidate is **not added to training automatically**.

In [8]:
teacher_prompt = (
    "Write one fictional observation in two sentences: civilians shopping at market stalls, "
    "a single camera, image degraded by fog, no independent corroboration. "
    "Do not include classification labels or add other events."
)
print(teacher_prompt)

Write one fictional observation in two sentences: civilians shopping at market stalls, a single camera, image degraded by fog, no independent corroboration. Do not include classification labels or add other events.


In [9]:
candidate = chat("base", [{"role": "user", "content": teacher_prompt}], max_tokens=180)
print("CANDIDATE — REVIEW BEFORE USE:\n")
print(candidate)

CANDIDATE — REVIEW BEFORE USE:

A lone vendor arranged bundles of dried chilies against a backdrop of blurred rooftops, the camera’s sensor struggling to resolve the shapes through a persistent veil of moisture. Shoppers moved in loose clusters, their faces obscured by the atmospheric haze, the single lens capturing only the ghostly silhouettes of the scene.


**Check the wording:** Is it still a market? Is the image degraded? Is there still only one source? Did the model invent anything?

If the wording changes those facts, revise or reject it. This tiny exercise checks label consistency and exact text overlap; it does not prove real-world coverage or semantic correctness.

### 6 · See what training scores
The model reads the instructions and observation as context. Training scores the **expected answer**.

Run the cell to inspect tokens at the context/answer boundary: `-100` is ignored by the loss; answer token IDs are scored. The tokenizer comes from the local cache—no weights are downloaded.

In [10]:
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download
checkpoint = snapshot_download('nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4', local_files_only=True)
tokenizer = AutoTokenizer.from_pretrained(checkpoint, local_files_only=True, trust_remote_code=True)
messages = training[0]['messages']
context = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True, enable_thinking=False)
full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
assert full_text.startswith(context)
encoded = tokenizer(full_text, add_special_tokens=False, return_offsets_mapping=True)
labels = [token if end > len(context) else -100
          for token, (_, end) in zip(encoded['input_ids'], encoded['offset_mapping'])]
assert any(label != -100 for label in labels)
token_counts = {'context_tokens_ignored': labels.count(-100),
                'answer_tokens_scored': sum(label != -100 for label in labels)}
# Inspect the boundary: token IDs remain aligned with labels, even across text boundaries.
boundary = next(i for i, label in enumerate(labels) if label != -100)
for i in range(max(0, boundary - 3), min(len(labels), boundary + 8)):
    print(i, repr(tokenizer.decode([encoded['input_ids'][i]])), 'label=', labels[i])
display(JSON(token_counts, expanded=True))

/home/nico/dev/nato-workshop/nato-workshop/.workshop/participant-repository/course/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Ignoring corrupted tree cache file /home/nico/.cache/huggingface/hub/models--nvidia--NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4/trees/bee7596271d1495f6992ae224aefde4410e816b8.json: [Errno 13] Permission denied: '/home/nico/.cache/huggingface/hub/models--nvidia--NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4/trees/bee7596271d1495f6992ae224aefde4410e816b8.json'


81 '\n' label= -100
82 '<think>' label= -100
83 '</think>' label= -100
84 '{"' label= 19227
85 'activity' label= 16570
86 '_type' label= 7532
87 '":' label= 2811
88 ' "' label= 1429
89 'c' label= 1099
90 'ivil' label= 7218
91 '_g' label= 11898


<IPython.core.display.JSON object>

### 7 · Grow the skill library
| Notebook | Skill | What it adds |
|---|---|---|
| 01 | `workshop-model-inventory` | Discover models and check their responses. |
| 02 | `workshop-training-data` | Build labeled data and check the result. |

The fixed reference skill gives OpenClaw three actions:

**`inspect-data` → `build-data` → `check-data`**

`build-data` runs the fixed lesson on the host CPU and calls Lightning for a fresh wording suggestion. Nano chooses the tools; it does not invent the labels or write arbitrary host code.

Open the actual skill below.

In [11]:
from html import escape
from IPython.display import HTML
skill = (ROOT / "course/agent/skills/workshop-training-data/SKILL.md").read_text()
display(HTML("<details><summary><b>Read SKILL.md</b></summary><pre style='white-space:pre-wrap'>"
             + escape(skill) + "</pre></details>"))

### 8 · Prepare the terminal demo
Run once for each new demo. This cell installs the growing skill library in a fresh sandbox workspace and starts a background host worker for this lesson’s three actions. **It does not ask the agent to do anything yet.**

Here, **installing a skill means placing its `SKILL.md` instruction file in OpenClaw’s workspace**. The current lesson’s source is `course/agent/skills/workshop-training-data/SKILL.md` in this repository.

Each run gets this structure inside the OpenShell sandbox (launcher files omitted):

```text
/home/sandbox/course-runs/openclaw-<run-id>/
├── openclaw.json                 # points OpenClaw at this run’s workspace
├── chapter_tool.py              # helper invoked by the agent
├── chapter.json                 # this lesson’s task and allowed actions
└── workspace/
    ├── AGENTS.md                 # workspace instructions
    └── skills/
        ├── workshop-training-data/
        │   └── SKILL.md
        └── ...                  # skills from earlier notebooks
```

OpenClaw discovers skills under `workspace/skills/`. The task you paste in the next step tells the agent which skill to read and supplies the full path to `chapter_tool.py` (the skill’s `<tool_path>` placeholder).

The agent invokes the helper through its execution tool. The helper forwards the selected action to the host worker, which runs the fixed lesson code and returns the result. Nano follows the skill and coordinates the calls.

This lesson does not require artifacts from an earlier OpenClaw demo.

Start the TUI task within **15 minutes** and keep this run’s `terminal_run` value for collecting results. If the worker expires, prepare a new run. The local run directory stores host results; the installed skills live at the sandbox paths printed below.


In [12]:
from course.agent.chapter_terminal import prepare_chapter_terminal, collect_chapter_terminal
from course.agent.notebook_terminal import open_terminal
terminal_run = prepare_chapter_terminal('02-training-data')
print("Local run directory:", terminal_run["directory"])
print("OpenClaw sandbox workspace:", terminal_run["remote"] + "/workspace")
print("Installed skill:", terminal_run["remote"] + "/workspace/skills/workshop-training-data/SKILL.md")
print("Python helper:", terminal_run["remote"] + "/chapter_tool.py")
print("\nThe agent will run these commands individually:")
for action in ['inspect-data', 'build-data', 'check-data']:
    print("python3 " + terminal_run["remote"] + "/chapter_tool.py " + action)

Local run directory: /home/nico/dev/nato-workshop/nato-workshop/.workshop/participant-repository/course/outputs/agent-demos/02-training-data/openclaw-ecdcf32682ef419d831e463bc321d194
OpenClaw sandbox workspace: /home/sandbox/course-runs/openclaw-ecdcf32682ef419d831e463bc321d194/workspace
Installed skill: /home/sandbox/course-runs/openclaw-ecdcf32682ef419d831e463bc321d194/workspace/skills/workshop-training-data/SKILL.md
Python helper: /home/sandbox/course-runs/openclaw-ecdcf32682ef419d831e463bc321d194/chapter_tool.py

The agent will run these commands individually:
python3 /home/sandbox/course-runs/openclaw-ecdcf32682ef419d831e463bc321d194/chapter_tool.py inspect-data
python3 /home/sandbox/course-runs/openclaw-ecdcf32682ef419d831e463bc321d194/chapter_tool.py build-data
python3 /home/sandbox/course-runs/openclaw-ecdcf32682ef419d831e463bc321d194/chapter_tool.py check-data


### 9 · Open OpenClaw and paste the task
Exit notebook 01’s TUI with `/exit` first. The next cell opens a terminal on the workshop computer’s desktop.

The TUI will say **Nano**, the lightweight orchestration model. The `build-data` tool calls **Lightning** for the wording suggestion.

Copy only the generated task below into the TUI. Watch the skill read and the three tool calls.

In [13]:
open_terminal(terminal_run)
print("\nPASTE THIS INTO OPENCLAW:\n")
print(terminal_run["prompt"])

Requested an OpenClaw terminal on the workshop computer’s desktop. When its TUI is ready, paste the task below.
If no window appears, in a Jupyter terminal run: bash /home/nico/dev/nato-workshop/nato-workshop/.workshop/participant-repository/course/outputs/agent-demos/02-training-data/openclaw-ecdcf32682ef419d831e463bc321d194/open-terminal.sh

PASTE THIS INTO OPENCLAW:

Use the workshop-training-data skill. First read /home/sandbox/course-runs/openclaw-ecdcf32682ef419d831e463bc321d194/workspace/skills/workshop-training-data/SKILL.md. The tool path is /home/sandbox/course-runs/openclaw-ecdcf32682ef419d831e463bc321d194/chapter_tool.py. Task: Build the example training data, check its labels, and show one fresh wording suggestion for review. The tool builds the lesson's labeled examples. Model-generated wording stays separate for review. Run inspect-data, then build-data, then check-data individually with exec elevated=false, yieldMs=10000, timeoutSeconds=180. If a command is still runnin

### 10 · Bring the results back
Wait until OpenClaw finishes, then run this cell. Look for **PASS**, **24 training examples**, **3 validation examples**, and a wording suggestion kept separate for review.

These results come from the agent’s own run folder, separate from your earlier manual examples. The check also verifies that the agent read its skill before running tools.

Running all cells without completing the TUI task will stop here; rerun this cell once the agent is finished.

In [14]:
agent_result = collect_chapter_terminal(terminal_run)
display(JSON(agent_result, expanded=True))

CHECK: PASS
{
  "training_examples": 24,
  "validation_examples": 3,
  "duplicate_observations": 0,
  "train_validation_overlap": 0,
  "supervised_tokens": 38,
  "teacher_suggestion": "A lone vendor arranged bundles of dried chilies against a backdrop of blurred rooftops, the camera\u2019s sensor struggling to resolve the shapes through a persistent veil of moisture. Shoppers moved in loose clusters, their faces obscured by the atmospheric haze, the single lens capturing only the ghostly silhouettes of the scene.",
  "review_status": "Not added to training; human review required"
}
Saved: /home/nico/dev/nato-workshop/nato-workshop/.workshop/participant-repository/course/outputs/agent-demos/02-training-data/openclaw-ecdcf32682ef419d831e463bc321d194


<IPython.core.display.JSON object>

**What you demonstrated:** define the answer → build examples → ask Lightning for wording → let Nano orchestrate the same work → check the evidence.

**Next:** attach a small adapter that can learn from these examples. Exit the TUI with `/exit` when done.